In [1]:
from pathlib import Path

#With NeXL successfully installed on Julia 1.12.7. 
#Open an additional terminal and enter in Jullia, with Julia 
#This Jupypter notebook imports the existing file Standards_df.csv. The  folder NeXL_XPP


JULIA_SCRIPT = r'''
using CSV
using DataFrames

const STANDARDS_PATH = length(ARGS) >= 1 ? ARGS[1] :
    "/Users/thomaslam/Documents/NeXL_XPP/Standards_df.csv"

const ATOMIC_WEIGHT = Dict(
    "O"  => 15.999,
    "Na" => 22.98976928,
    "Mg" => 24.305,
    "Al" => 26.9815385,
    "Si" => 28.085,
    "K"  => 39.0983,
    "Ca" => 40.078,
    "Ti" => 47.867,
    "Mn" => 54.938044,
    "Fe" => 55.845
)

const OXIDE_DEFINITIONS = Dict(
    :SiO2  => ("Si", 1, 2),
    :TiO2  => ("Ti", 1, 2),
    :Al2O3 => ("Al", 2, 3),
    :FeO   => ("Fe", 1, 1),
    :Fe2O3 => ("Fe", 2, 3),
    :MnO   => ("Mn", 1, 1),
    :MgO   => ("Mg", 1, 1),
    :CaO   => ("Ca", 1, 1),
    :Na2O  => ("Na", 2, 1),
    :K2O   => ("K", 2, 1)
)

function oxide_row(standards_df::DataFrame, standard_id::AbstractString)
    matches = filter(
        row -> strip(String(row.standard_id)) == standard_id,
        standards_df
    )

    nrow(matches) == 1 || error(
        "Expected exactly one $standard_id row; found $(nrow(matches))."
    )

    return matches[1, :]
end

function elemental_mass_fractions(row::DataFrameRow)
    elemental_wtpercent = Dict{String, Float64}()

    for (oxide, (element, n_element, n_oxygen)) in OXIDE_DEFINITIONS
        oxide in propertynames(row) || continue

        oxide_wtpercent = row[oxide]

        if ismissing(oxide_wtpercent) || Float64(oxide_wtpercent) <= 0.0
            continue
        end

        oxide_wtpercent = Float64(oxide_wtpercent)

        oxide_molar_mass =
            n_element * ATOMIC_WEIGHT[element] +
            n_oxygen * ATOMIC_WEIGHT["O"]

        elemental_wtpercent[element] =
            get(elemental_wtpercent, element, 0.0) +
            oxide_wtpercent * n_element * ATOMIC_WEIGHT[element] /
            oxide_molar_mass

        elemental_wtpercent["O"] =
            get(elemental_wtpercent, "O", 0.0) +
            oxide_wtpercent * n_oxygen * ATOMIC_WEIGHT["O"] /
            oxide_molar_mass
    end

    isempty(elemental_wtpercent) && error(
        "No recognized positive oxide values found for $(row.standard_id). " *
        "Available columns: " * join(string.(propertynames(row)), ", ")
    )

    total = sum(values(elemental_wtpercent))
    total > 0.0 || error("Total elemental mass is not positive.")

    return Dict{String, Float64}(
        element => value / total for (element, value) in elemental_wtpercent
    )
end

function print_fractions(standard_id::AbstractString,
                         fractions::Dict{String, Float64})
    println("\n$standard_id elemental mass fractions")
    println("-" ^ 40)

    for (element, fraction) in sort(collect(fractions); by = first)
        println(rpad(element, 3), "  ", round(fraction; digits = 8))
    end

    println("-" ^ 40)
    println("Total = ", round(sum(values(fractions)); digits = 8))
end

function main()
    standards_df = CSV.read(STANDARDS_PATH, DataFrame)

    for standard_id in (
        "albite_spi_mci",
        "MgO_spi_mci",
        "orthoclase_spi_mci",
        "wollastonite_spi_mci",
        "rhodonite_spi_mci",
        "hematite_spi_mci"
    )
        row = oxide_row(standards_df, standard_id)
        fractions = elemental_mass_fractions(row)
        print_fractions(standard_id, fractions)
    end
end

main()
'''

Path("standards_fractions.jl").write_text(JULIA_SCRIPT, encoding="utf-8")
print("Wrote standards_fractions.jl")

Wrote standards_fractions.jl


In [2]:
import subprocess
from pathlib import Path

STANDARDS_CSV = Path(
    "/Users/thomaslam/Documents/NeXL_XPP/Standards_df.csv"
)

result = subprocess.run(
    ["julia", "standards_fractions.jl", str(STANDARDS_CSV)],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print("Julia failed:")
    print(result.stderr)


albite_spi_mci elemental mass fractions
----------------------------------------
Al   0.10395742
Ca   0.001634
K    0.00140284
Na   0.08613272
O    0.4873805
Si   0.31949253
----------------------------------------
Total = 1.0

MgO_spi_mci elemental mass fractions
----------------------------------------
Mg   0.60304188
O    0.39695812
----------------------------------------
Total = 1.0

orthoclase_spi_mci elemental mass fractions
----------------------------------------
Al   0.08864193
Fe   0.01406559
K    0.12865476
Na   0.00675432
O    0.45883288
Si   0.30305051
----------------------------------------
Total = 1.0

wollastonite_spi_mci elemental mass fractions
----------------------------------------
Ca   0.34502066
O    0.41319851
Si   0.24178083
----------------------------------------
Total = 1.0

rhodonite_spi_mci elemental mass fractions
----------------------------------------
Ca   0.03309042
Fe   0.0303553
Mg   0.01175932
Mn   0.32759742
O    0.37853066
Si   0.21866689
----